In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
# PARAMETERS
BRONZE_CONTAINER_FILES = [r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-001\container\ams__container_2018__202001290000_part_0.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-001\container\ams__container_2018__202001290000_part_2.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-001\container\ams__container_2018__202001290000_part_3.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-002\container\ams__container_2018__202001290000_part_1.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2019\container-20260331T184903Z-3-001\container\ams__container_2019__202001080000_part_0.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2019\container-20260331T184903Z-3-001\container\ams__container_2019__202001080000_part_2.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2019\container-20260331T184903Z-3-001\container\ams__container_2019__202001080000_part_3.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2019\container-20260331T184903Z-3-002\container\ams__container_2019__202001080000_part_1.csv']
SILVER_CONTAINER_OUTPUT_PATH = r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Silver-Layer\container_silver.csv'
CONTAINER_PRIMARY_KEY = "container_number"
CHUNK_SIZE = 100000

In [5]:
def load_container_data_from_multiple_files(file_paths, chunk_size=100000) -> pd.DataFrame:
    all_chunks = []

    for file_path in file_paths:
        print(f"Reading: {file_path}")

        for chunk in pd.read_csv(
            file_path,
            chunksize=chunk_size,
            dtype=str,
            low_memory=False
        ):
            all_chunks.append(chunk)

    if not all_chunks:
        raise ValueError("No data loaded from the provided files.")
    
    return pd.concat(all_chunks, ignore_index=True)

containerdf = load_container_data_from_multiple_files(BRONZE_CONTAINER_FILES, chunk_size=CHUNK_SIZE)
print("Container data loaded successfully.")
    

Reading: C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-001\container\ams__container_2018__202001290000_part_0.csv
Reading: C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-001\container\ams__container_2018__202001290000_part_2.csv
Reading: C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-001\container\ams__container_2018__202001290000_part_3.csv
Reading: C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-002\container\ams__container_2018__202001290000_part_1.csv
Reading: C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2019\container-20260331T184903Z-3-001\container\ams__container_2019__202001080000_part_0.csv
Reading: C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2019\container-20260331T184903Z-3-001\container\ams__container_2019__202001080000_part_2.csv
Reading: C:\User

In [6]:
print(containerdf.columns.tolist())

['identifier', 'container_number', 'seal_number_1', 'seal_number_2', 'equipment_description_code', 'container_length', 'container_height', 'container_width', 'container_type', 'load_status', 'type_of_service']


In [7]:
required_columns = ["identifier", "container_number", "seal_number_1", "seal_number_2", "equipment_description_code", "container_length", "container_height", "container_width", "container_type", "load_status", "type_of_service"]

def validate_required_columns(df, required_columns):
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    
validate_required_columns(containerdf, required_columns)
print("All required columns are present.")


All required columns are present.


In [8]:
def standardize_string_columns(df, cols):

    replacements = {
        "": pd.NA,
        "null": pd.NA,
        "NULL": pd.NA,
        "NaN": pd.NA,
        "nan": pd.NA,
        "N/A": pd.NA,
        "n/a": pd.NA,
    }

    for col in cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .replace(replacements)
            )
    return df

container_string_cols = ["container_number", "seal_number_1", "seal_number_2", "equipment_description_code", "container_type", "load_status", "type_of_service"]

standardize_string_columns(containerdf, container_string_cols)
print("String columns standardized.")


String columns standardized.


In [9]:
def convert_container_numeric_columns(df):
    numeric_cols = ["container_length", "container_height", "container_width"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce", downcast="float")
    return df

convert_container_numeric_columns(containerdf)
print("Numeric columns converted successfully.")

Numeric columns converted successfully.


In [11]:
def clean_invalid_dimesnions(df):
    dim_cols = ["container_length", "container_height", "container_width"]
    mask = df[dim_cols].sum(axis=1) == 0
    df.loc[mask, dim_cols] = pd.NA
    return df

clean_invalid_dimesnions(containerdf)
print("Invalid dimension rows cleaned.")

Invalid dimension rows cleaned.


In [12]:
def remove_duplicates(df):
    if {"identifier", CONTAINER_PRIMARY_KEY} <= set(df.columns):
        df.drop_duplicates(
            subset=["identifier", CONTAINER_PRIMARY_KEY],
            keep="first",
            inplace=True
        )
    return df

remove_duplicates(containerdf)
print("Duplicate rows removed based on identifier and container_number.")

Duplicate rows removed based on identifier and container_number.


In [13]:
def build_silver_container(df):
    
    silver_cols = [
        "identifier",
        "container_number",
        "seal_number_1",
        "seal_number_2",
        "equipment_description_code",
        "container_type",
        "container_length",
        "container_width",
        "container_height",
        "container_volume",
        "load_status",
        "type_of_service"
    ]

    existing_cols = [c for c in silver_cols if c in df.columns]
    return df.loc[:, existing_cols]

silver_containerdf = build_silver_container(containerdf)
print("Silver container dataframe built with selected columns.")


Silver container dataframe built with selected columns.


In [14]:
def save_silver_container(df, output_path):
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_file, index=False)

save_silver_container(silver_containerdf, SILVER_CONTAINER_OUTPUT_PATH)
print(f"Silver container data saved to {SILVER_CONTAINER_OUTPUT_PATH}.")

Silver container data saved to C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Silver-Layer\container_silver.csv.
